### Cross-vehicle consistency checks for NTD cases

The *baseline* scenario for neural tube defect (NTD) cases should not depend on vehicle for each country. This notebook verifies two properties of the NTD model outputs (`ntd_cases_by_scenario.csv`) for each country:

**Check 1: Zero scenario equals baseline when baseline effective coverage is zero**
- For each country/vehicle pair, we load the baseline effective folate coverage from `0100_data_prep`.
- If baseline coverage is 0 (no vehicle fortification at baseline), then the `zero` and `baseline` NTD cases must be identical. We assert this and print a confirmation.

**Check 2: Baseline NTD cases agree across vehicles within a country**
- The first vehicle's results for each country are used as a reference. Every other vehicle's `baseline` NTD cases must match that reference.
- The check is skipped for India because baseline rice fortification coverage is added on top of the GBD estimates, so India's baseline differs between vehicles. This should be revisited so baseline handling is consistent across vehicles.

Parameterized: `results_ref` is the git ref to read the NTD results and baseline coverage CSVs
from (None = working tree, the default for reruns after a pipeline run). Neither input is
archived to the team drive, so a model iteration is identified by its commit, not by
`MODEL_NUMBER`.

In [10]:
results_ref = "91bb8a1"  # git ref for the NTD and coverage CSVs; None reads the working tree

In [11]:
import io
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parents[2]
NTD_RESULTS = "0500_neural_tube_defects_model/results"
COVERAGE = "0100_data_prep/results/folate"


def list_ntd_csvs(ref):
    """`{country}/{vehicle}/ntd_cases_by_scenario.csv` under NTD_RESULTS; the nested
    intervention_* subfolders are skipped."""
    if ref is None:
        paths = [str(p.relative_to(REPO)) for p in (REPO / NTD_RESULTS).glob("*/*/ntd_cases_by_scenario.csv")]
    else:
        paths = subprocess.run(["git", "ls-tree", "-r", "--name-only", ref, "--", NTD_RESULTS],
                               capture_output=True, text=True, cwd=REPO, check=True).stdout.split()
    depth = len(Path(NTD_RESULTS).parts) + 3
    return sorted(p for p in paths if p.endswith("/ntd_cases_by_scenario.csv") and len(Path(p).parts) == depth)


def read_csv(ref, path):
    if ref is None:
        return pd.read_csv(REPO / path)
    out = subprocess.run(["git", "show", f"{ref}:{path}"],
                         capture_output=True, text=True, cwd=REPO, check=True)
    return pd.read_csv(io.StringIO(out.stdout))


ntd = {}
for p in list_ntd_csvs(results_ref):
    country, vehicle = Path(p).parts[-3:-1]
    ntd[(country, vehicle)] = read_csv(results_ref, p)
coverage = {
    (country, vehicle): read_csv(results_ref, f"{COVERAGE}/{vehicle}/baseline_fortification/effective_coverage/{country}.csv")
    for country, vehicle in ntd
}
arms = pd.DataFrame(sorted(ntd), columns=["country", "vehicle"])
print(f"{len(ntd)} arms loaded from {results_ref or 'the working tree'}")
arms

8 arms loaded from 91bb8a1


,country,vehicle
0,ethiopia,salt
1,ethiopia,wheat
2,india,rice
3,india,wheat
4,nigeria,bouillon
5,nigeria,rice
6,nigeria,salt
7,nigeria,wheat


In [12]:
for country in arms.country.unique():
    vehicles = arms[arms.country == country].vehicle.tolist()
    # india's baseline rice coverage is added on top of GBD estimates, so rice is excluded from the cross-vehicle baseline check there
    comparable_vehicles = [v for v in vehicles if not (country == "india" and v == "rice")]
    first_vehicle_values = ntd[(country, comparable_vehicles[0])]
    first_vehicle_values = first_vehicle_values[first_vehicle_values.scenario.isin(["zero", "baseline"])]
    first_vehicle_values = first_vehicle_values.set_index([d for d in first_vehicle_values.columns if d != "value"]).value

    for vehicle in vehicles:
        baseline = coverage[(country, vehicle)]
        baseline = baseline.set_index([c for c in baseline.columns if c != "value"]).value
        
        vehicle_values = ntd[(country, vehicle)]
        vehicle_values = vehicle_values[vehicle_values.scenario.isin(["zero", "baseline"])]
        vehicle_values = vehicle_values.set_index([d for d in vehicle_values.columns if d != "value"]).value

        # no partial check exists if only some quintiles are 0
        if (baseline == 0).all():
            assert (vehicle_values.xs("zero", level = "scenario") == vehicle_values.xs("baseline", level = "scenario")).all()
            print(f"Zero and baseline matched for {country}/{vehicle}.")
        else:
            print(f"Baseline fortification exists for {country}/{vehicle}.")

        # this check doesn't pass in india due to baseline rice coverage adding to GBD estimates; this should be revisted for consistency in baseline across vehicles
        if vehicle in comparable_vehicles:
            assert np.allclose(
                vehicle_values.xs("baseline", level = "scenario"), 
                first_vehicle_values.xs("baseline", level = "scenario"),
                rtol = 0,
                atol = 0.000001
            )
    skipped = sorted(set(vehicles) - set(comparable_vehicles))
    print(f"Baseline cases matched for {', '.join(comparable_vehicles)} in {country}."
          + (f" Skipped: {', '.join(skipped)}." if skipped else ""))
        

Zero and baseline matched for ethiopia/salt.
Baseline fortification exists for ethiopia/wheat.
Baseline cases matched for salt, wheat in ethiopia.
Baseline fortification exists for india/rice.
Baseline fortification exists for india/wheat.
Baseline cases matched for wheat in india. Skipped: rice.
Zero and baseline matched for nigeria/bouillon.
Zero and baseline matched for nigeria/rice.
Zero and baseline matched for nigeria/salt.
Zero and baseline matched for nigeria/wheat.
Baseline cases matched for bouillon, rice, salt, wheat in nigeria.
